# Train Clarion RFI Drafting Model (Qwen1.5)

This notebook uses [Unsloth](https://github.com/unslothai/unsloth) to fine-tune a small Qwen1.5 model on our synthetic RFI dataset.

**Instructions:**
1. Ensure you are running this in Google Colab with a GPU (Runtime -> Change runtime type -> T4 GPU).
2. Upload the `synthetic_rfis.jsonl` file generated locally to the `/content/` folder in Colab.

In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes transformers==4.43.2


In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None # None for auto detection
load_in_4bit = True

# Load Qwen1.5 model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen1.5-1.8B-Chat",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0, 
    bias = "none",    
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

In [ ]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "chatml", # Qwen uses ChatML formatting
    mapping = {"role" : "role", "content" : "content", "user" : "user", "assistant" : "assistant"}
)

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
    return { "text" : texts }

from datasets import load_dataset

# Load our synthetic dataset (uploaded to /content)
dataset = load_dataset("json", data_files="/content/synthetic_rfis.jsonl", split="train")
dataset = dataset.map(formatting_prompts_func, batched = True)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()

In [ ]:
# Export to GGUF format for Ollama usage locally
# This will create a 4-bit quantized GGUF file we can download.
model.save_pretrained_gguf("clarion_qwen_1.5_gguf", tokenizer, quantization_method = "q4_k_m")

print("Training complete and model exported. You can now download the folder clarion_qwen_1.5_gguf")